# What gets better with size, and what does not

MichAl Academy, unit 4.8.

Run each cell with **Shift+Enter**.

Models are said to gain abilities suddenly as they get bigger: nothing, nothing,
nothing, then the thing works. This notebook builds that effect from scratch on a
task small enough to train on a laptop, and then scores the same models a second
way to show what the first way was hiding.

The task is the pointer chase from unit 4.3.3. Sixteen slots, each holding a
number that names another slot; follow the chain five times and report where you
land. Tables are drawn fresh for every batch, so nothing can be memorised.


In [ ]:
import time
import warnings

import numpy as np
import torch
from torch import nn

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

SLOTS, HOPS = 16, 5
QUERY, EQ, END = SLOTS, SLOTS + 1, SLOTS + 2
VOCAB = SLOTS + 3
PREFIX = SLOTS + 3               # the table, a marker, the start slot, '='
MAXLEN = PREFIX + 2


def make_batch(n, rng):
    """n sequences: the table, a marker, the start slot, '=', the answer, END."""
    table = rng.integers(0, SLOTS, (n, SLOTS))
    start = rng.integers(0, SLOTS, n)
    cur = start
    for _ in range(HOPS):
        cur = table[np.arange(n), cur]
    seq = np.concatenate([
        table,
        np.full((n, 1), QUERY),
        start[:, None],
        np.full((n, 1), EQ),
        cur[:, None],
        np.full((n, 1), END),
    ], axis=1)
    return seq, cur


seq, ans = make_batch(2, np.random.default_rng(0))
print("sequence", seq[0])
print("answer  ", ans[0], f"  (chance is 1/{SLOTS} = {1 / SLOTS:.4f})")


## A stack of models, identical except for width

`d` is how many numbers each position carries. Depth, data, training steps,
learning rate and seed are all held fixed. **Width is the only thing that
changes.**


In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads=4):
        super().__init__()
        self.heads = heads
        self.q, self.k, self.v = (nn.Linear(d, d) for _ in range(3))
        self.proj = nn.Linear(d, d)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))

    def forward(self, x, mask):
        h = self.n1(x)
        B, L, d = h.shape
        dh = d // self.heads
        shape = (B, L, self.heads, dh)
        q = self.q(h).view(shape).transpose(1, 2)
        k = self.k(h).view(shape).transpose(1, 2)
        v = self.v(h).view(shape).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / dh ** 0.5
        s = s.masked_fill(mask, float("-inf")).softmax(dim=-1)
        x = x + self.proj((s @ v).transpose(1, 2).reshape(B, L, d))
        return x + self.mlp(self.n2(x))


class Model(nn.Module):
    def __init__(self, d, blocks=6, heads=4):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(MAXLEN, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(blocks)])
        self.norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, VOCAB)
        self.register_buffer("mask", torch.triu(
            torch.ones(MAXLEN, MAXLEN, dtype=torch.bool), 1))

    def forward(self, x):
        L = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(L))
        for b in self.blocks:
            h = b(h, self.mask[:L, :L])
        return self.out(self.norm(h))


loss_fn = nn.CrossEntropyLoss()


def train_and_score(d, steps=2500, seed=0, n=64):
    torch.manual_seed(seed)
    m = Model(d)
    rng = np.random.default_rng(seed)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(steps):
        seq, _ = make_batch(n, rng)
        x = torch.tensor(seq)
        opt.zero_grad()
        logits = m(x[:, :-1])
        target = x[:, 1:].clone()
        target[:, :PREFIX - 1] = -100        # score the answer token only
        loss_fn(logits.reshape(-1, VOCAB), target.reshape(-1)).backward()
        opt.step()

    m.eval()
    with torch.no_grad():
        seq, ans = make_batch(2000, np.random.default_rng(999))
        x = torch.tensor(seq)
        at = m(x[:, :-1])[:, PREFIX - 1, :]   # scores for the answer position
        yt = torch.tensor(ans)
        acc = (at.argmax(-1) == yt).float().mean().item()
        # Probability put on the right answer, whether or not it was the top one.
        # Same models, same test, a measure that does not round to pass or fail.
        p_right = at.softmax(-1).gather(1, yt[:, None])[:, 0].mean().item()
    return acc, p_right, sum(p.numel() for p in m.parameters())


WIDTHS = (8, 12, 16, 20, 24, 32)
rows = []
t0 = time.time()
print(f"{'d':>4}{'params':>10}{'accuracy':>10}{'P(right)':>10}")
for d in WIDTHS:
    acc, pr, n_par = train_and_score(d)
    rows.append((d, n_par, acc, pr))
    print(f"{d:>4}{n_par:>10,}{acc:>10.4f}{pr:>10.4f}")
print(f"({time.time() - t0:.0f}s)")


## The same models, read two ways

Accuracy is the column people quote, and it throws away everything except whether
the top answer was right. `P(right)` keeps the rest: how much probability the
model put on the correct slot, whether or not it came first.


In [ ]:
print(f"{'d':>4}  {'accuracy':>9}  {'step':>8}   {'P(right)':>9}  {'step':>8}")
for i, (d, n_par, acc, pr) in enumerate(rows):
    if i == 0:
        print(f"{d:>4}  {acc:>9.4f}  {'':>8}   {pr:>9.4f}  {'':>8}")
    else:
        print(f"{d:>4}  {acc:>9.4f}  {acc - rows[i-1][2]:>+8.4f}   "
              f"{pr:>9.4f}  {pr - rows[i-1][3]:>+8.4f}")

accs = [r[2] for r in rows]
prs = [r[3] for r in rows]
sa = [accs[i] - accs[i-1] for i in range(1, len(accs))]
sp = [prs[i] - prs[i-1] for i in range(1, len(prs))]
print()
print(f"accuracy: largest single step {max(sa):+.4f}, "
      f"{max(sa) / sum(sa):.0%} of the whole climb")
print(f"P(right): largest single step {max(sp):+.4f}, "
      f"{max(sp) / sum(sp):.0%} of the whole climb")


## What this unit measured

- **The same models give a cliff or a slope depending on how you score them.**
  Accuracy is pass-or-fail per question, so a model steadily getting better shows
  nothing until it crosses the line and then appears to arrive all at once.
- **Nothing about the models is discontinuous.** They differ only in width, and
  were trained identically.
- **This is worth knowing before reading a capability claim.** A benchmark scored
  pass-or-fail will report sudden abilities. The same benchmark scored on
  probability usually will not.
